In [ ]:
# 데이터 다운로드
!gdown 10bnEC6-ZfXZFZ2mb3zoWd38TjYufanWo

^C


In [ ]:
# 라이브러리 로드
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torchvision.models as models, datasets
import torch.nn.functional as F
from torchvision import datasets
from torch import nn, optim
from torchvision.transforms import v2, InterpolationMode

In [ ]:
# Config(CPU 환경에서 실행)

# ngpus = 8
# batch_size = 32

epochs=15
opt='sgd'
momentum=0.9

# 파인튜닝으로 학습률을 더 줄임
lr=0.001
lr_scheduler='steplr'
lr_step_size=5
lr_gamma=0.1


# Regularization
weight_decay=1e-4


# Resizing
interpolation=InterpolationMode.BILINEAR
val_resize_size=256
val_crop_size=224
train_crop_size=224

In [ ]:
# 난수 고정
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# Cuda설정
# torch.cuda.manual_seed(42)
# torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 1. 토치비전 v2 전처리 파이프라인 정의
train_transforms = v2.Compose([
    v2.ToImage(),   # 이미지 > 데이터
    v2.RandomResizedCrop(size=(224,224), interpolation=InterpolationMode.BILINEAR, antialias=True), # 무작위로 잘라낸 후 크기 수정
    v2.RandomHorizontalFlip(0.5),   # 절반의 이미지는 좌우반전
    v2.ToDtype(torch.float, scale=True),    # 데이터 타입을 텐서로 바꾸고 범위를 [0, 1]로 조절
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),    # 이미지넷 가중치
    v2.ToPureTensor()   # 메타데이터 제거
])

val_transforms = v2.Compose([
    v2.ToImage(),
    v2.Resize(size=(256,256), interpolation=InterpolationMode.BILINEAR, antialias=True),
    v2.CenterCrop(size=(224,224)),
    v2.ToDtype(torch.float, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    v2.ToPureTensor()
])

# 2. ImageFolder를 이용한 데이터셋 로드
data_dir = "./Pistachio_Image_Dataset" 
dataset = datasets.ImageFolder(root=data_dir)

# 3. 학습용(80%) / 검증용(20%) 데이터 분할
generator = torch.Generator().manual_seed(42) # 결과 재현을 위한 랜덤 시드 고정
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_indices, val_indices = random_split(
    range(len(dataset)),
    [train_size, val_size],
    generator=generator
)

train_dataset = Subset(
    datasets.ImageFolder(root=data_dir, transform=train_transforms),
    train_indices.indices
)
val_dataset = Subset(
    datasets.ImageFolder(root=data_dir, transform=val_transforms),
    val_indices.indices
)

# 4. DataLoader 생성
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False,)

# 5. 로드 결과 및 첫 번째 배치의 텐서 타입 검증
images, labels = next(iter(train_loader))
print(f"총 이미지 개수: {len(dataset)} | 클래스 종류: {dataset.classes}")
print(f"이미지 배치 크기: {images.shape} (타입: {images.dtype})")
print(f"라벨 배치 크기: {labels.shape}")


총 이미지 개수: 2148 | 클래스 종류: ['Kirmizi_Pistachio', 'Siirt_Pistachio']
이미지 배치 크기: torch.Size([4, 3, 224, 224]) (타입: torch.float32)
라벨 배치 크기: torch.Size([4])


In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

# ResNet50 파인튜닝
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)
print("모델 전처리 과정 : \n", weights.transforms())

모델 전처리 과정 : 
 ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(),
    lr=lr,
    momentum=momentum,
    weight_decay=weight_decay,
)
# 분류(CE), SGD(모멘텀, 가중치 감쇠)

In [ ]:
scheduler = optim.lr_scheduler.StepLR(
    optimizer=optimizer,
    step_size=lr_step_size,
    gamma=lr_gamma,
)
# 스텝 사이즈마다 gamma만큼 lr조정

In [8]:
model = model.to(device)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [10]:
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

    acc = 100. * correct / total
    return total_loss / len(loader), acc

In [ ]:
epochs = 15
for t in range(epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    scheduler.step()
    val_loss, val_acc = validate(model, val_loader, criterion)

    print(f"[Epoch {t+1}/{epochs}] "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.2f}%")
print("Done!")
# CPU 환경에서 약 70분 소요

[Epoch 1/15] Train Loss: 0.8260 | Val Loss: 0.5337 | Val Acc: 72.56%
[Epoch 2/15] Train Loss: 0.3701 | Val Loss: 0.1569 | Val Acc: 95.12%
[Epoch 3/15] Train Loss: 0.3146 | Val Loss: 0.1858 | Val Acc: 92.79%
[Epoch 4/15] Train Loss: 0.2939 | Val Loss: 0.2457 | Val Acc: 90.00%
[Epoch 5/15] Train Loss: 0.2634 | Val Loss: 0.1991 | Val Acc: 92.33%
[Epoch 6/15] Train Loss: 0.2026 | Val Loss: 0.0681 | Val Acc: 98.14%
[Epoch 7/15] Train Loss: 0.1932 | Val Loss: 0.0500 | Val Acc: 98.84%
[Epoch 8/15] Train Loss: 0.1758 | Val Loss: 0.0354 | Val Acc: 99.53%
[Epoch 9/15] Train Loss: 0.1590 | Val Loss: 0.0387 | Val Acc: 99.53%
[Epoch 10/15] Train Loss: 0.1691 | Val Loss: 0.0588 | Val Acc: 98.37%
[Epoch 11/15] Train Loss: 0.1571 | Val Loss: 0.0340 | Val Acc: 99.53%
[Epoch 12/15] Train Loss: 0.1701 | Val Loss: 0.0340 | Val Acc: 99.77%
[Epoch 13/15] Train Loss: 0.1387 | Val Loss: 0.0382 | Val Acc: 99.77%
[Epoch 14/15] Train Loss: 0.1568 | Val Loss: 0.0344 | Val Acc: 99.77%
[Epoch 15/15] Train Loss: 0.1

In [ ]:
# 모델 저장
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth
